# TML — Kaggle T4×2 RapidOCR Worker
Dual-GPU worker for TennisMyLife PRE-AI. It uses a dedicated VPS claim and writes OCR results back by SFTP.

Start with `MODE='BENCHMARK'`: the 40-page benchmark writes to an isolated cache and cannot interfere with production. After validation switch to `MODE='PRODUCTION'`.

**Kaggle settings:** Accelerator = **GPU T4 x2**, Internet = **On**.


In [ ]:
YEAR = 1905
MODE = 'BENCHMARK'  # BENCHMARK or PRODUCTION
WORKERS_PER_GPU = 3
DOWNLOADERS_PER_GPU = 4
VPS_HOST = 'vibrant-lovelace.82-165-11-122.plesk.page'
VPS_USER = 'andre'
VPS_PORT = 2222
BASE = f'/home/andre/GallicaJobs/gallica-{YEAR}-all-tennis/GALlica_{YEAR}_ALL_TENNIS'
CLAIM = f'{BASE}/00_MANIFEST/kaggle_benchmark_claim.tsv' if MODE == 'BENCHMARK' else f'{BASE}/00_MANIFEST/kaggle_active_claims.tsv'
STOP = f'{BASE}/00_MANIFEST/kaggle_rapid_stop_{YEAR}.flag'
print('CONFIG', YEAR, MODE, CLAIM)


In [ ]:
import os, subprocess, sys
REPO='/kaggle/working/Tennis-OCR-Pipeline'
if os.path.isdir(REPO): subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
else: subprocess.run(['git','clone','-q','https://github.com/Tennismylife/Tennis-OCR-Pipeline.git',REPO],check=True)
subprocess.run([sys.executable,'-m','pip','uninstall','-y','onnxruntime','onnxruntime-gpu'],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',f'{REPO}/colab/requirements.txt'],check=True)
subprocess.run(['nvidia-smi','-L'],check=True)
import onnxruntime as ort
print('ORT',ort.__version__,'providers',ort.get_available_providers())
assert 'CUDAExecutionProvider' in ort.get_available_providers(), 'CUDAExecutionProvider unavailable'


In [ ]:
import base64, os, subprocess
KEY_FILE='/kaggle/working/tml_kaggle_key'
key_b64=None
try:
    from kaggle_secrets import UserSecretsClient
    key_b64=UserSecretsClient().get_secret('TML_VPS_SSH_KEY_B64')
except Exception:
    pass
if key_b64:
    open(KEY_FILE,'wb').write(base64.b64decode(key_b64)); os.chmod(KEY_FILE,0o600)
    print('SSH_KEY_READY from Kaggle Secret')
else:
    if not os.path.exists(KEY_FILE):
        subprocess.run(['ssh-keygen','-t','ed25519','-N','','-f',KEY_FILE,'-C','tml-kaggle-worker'],check=True,stdout=subprocess.DEVNULL)
    print('PUBLIC KEY — send ONLY this line to ChatGPT for VPS authorization:')
    print(open(KEY_FILE+'.pub').read().strip())
    print('Keep this Kaggle session open; the private key remains only in /kaggle/working.')


In [ ]:
import base64, subprocess, sys
key_b64=base64.b64encode(open(KEY_FILE,'rb').read()).decode()
cmd=[sys.executable,'-u',f'{REPO}/kaggle/dual_t4_watch.py',
     '--vps-host',VPS_HOST,'--vps-user',VPS_USER,'--vps-port',str(VPS_PORT),
     '--vps-key-b64',key_b64,'--claim',CLAIM,'--stop-flag',STOP,
     '--workers-per-gpu',str(WORKERS_PER_GPU),'--downloaders-per-gpu',str(DOWNLOADERS_PER_GPU),'--poll','10']
print('STARTING KAGGLE T4x2 WATCHER', MODE)
subprocess.run(cmd,check=True)
